# QSNN v3 — publication-grade attribution study

**Datasets**
- **CIFAR-10**: 10 classes, RGB 32×32; 45,000 train / 5,000 validation (seeded hold-out) / 10,000 test.
- **MNIST** (Part C, replication of Xu et al. PPF-SQNN): 10 classes, 28×28; 55,000 train / 5,000 validation / 10,000 test.

**What v3 adds over v2**

| # | Change | Where |
|---|---|---|
| 1 | End-to-end training (encoder + head jointly); removes the frozen-encoder bias | Stage 4 |
| 2 | TOST equivalence tests (±0.5 pts): "no difference" becomes a positive claim | `make_paper_tables.py` |
| 3 | Per-image test predictions saved; paired bootstrap CIs over test images and McNemar tests | all stages |
| 4 | Learning rate tuned per variant on validation (grid {3e-4, 1e-3, 3e-3, 1e-2}, separate tuning seeds) | all stages |
| 5 | Part C completed: per-branch accuracy, reduction-width sweep (5/20/60), logit fusion, validation-selected ξ | Stage 5 |
| 6 | 10 evaluation seeds (heads, Part C); 3 seeds end-to-end | all stages |
| 7 | Stronger encoder (BatchNorm, 4 conv layers, augmentation, direct input coding): tests the null result at higher accuracy | Stage 3 |
| + | Deterministic algorithms, environment log, saved split indices, resumable runs | Stage 0 |

**Statistics** (computed by `make_paper_tables.py`, embedded in Stage 6 and also usable stand-alone): paired t-tests with Holm correction within each experiment, Wilcoxon, TOST equivalence, paired bootstrap 95% CI over test images, per-seed McNemar. Verdicts are **BETTER / WORSE / EQUIVALENT / INCONCLUSIVE**.

**Running on Kaggle (GPU, Internet on).** The full study does not fit in one 12-hour session. Every run is appended to `runs_*.csv` as soon as it finishes, and finished runs are skipped on restart.
1. Run with the stage flags you want → *Save Version (Save & Run All)*.
2. Next session: add that version's output as an input and set `CFG['resume_from']` to its `qsnn_v3_results` folder; it is copied in and completed runs are skipped.
3. Point `prev_features_4` / `prev_features_16` at your earlier frozen features to skip retraining those encoders.

Approximate GPU cost (P100/T4): Stage 2 heads ≈ 3–5 h · Stage 3 strong encoder ≈ 1.5 h + heads ≈ 2 h · Stage 4 end-to-end ≈ 35–45 min per run (21 runs by default) · Stage 5 MNIST ≈ 3–4 h.

Set `SMOKE_TEST = True` to check the whole pipeline in a few minutes on CPU (random data; results meaningless).

## Stage 0 — setup, determinism, environment log

In [ ]:
# !pip install -q snntorch "qiskit>=1.0,<2.0"

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')     # needed for deterministic cuBLAS
import sys, math, copy, time, random, json, shutil, itertools, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
import torchvision.transforms as T
import snntorch as snn
from snntorch import surrogate, utils as snn_utils
from scipy import stats
import matplotlib.pyplot as plt

SMOKE_TEST = False

CFG = dict(
    data_root='./data', out='./qsnn_v3_results', resume_from=None,
    # stages
    run_diagnostics=True, run_heads=True, run_strong=True, run_e2e=True, run_partC=True, run_aggregate=True,
    # seeds and tuning
    eval_seeds=list(range(10)), tune_seeds=[100, 101], tune_lr=True, lr_grid=[3e-4, 1e-3, 3e-3, 1e-2], default_lr=1e-3,
    # base encoder (as in v1/v2)
    num_steps=25, beta=0.9, dropout=0.3, enc_batch=128, enc_lr=1e-3, enc_epochs=30, enc_patience=6,
    prev_features_4='./attribution_results/frozen_angles.pt',
    prev_features_16='./qsnn_v2_results/frozen_angles_16.pt',
    # strong encoder (item 7)
    strong_steps=6, strong_epochs=40, strong_lr=1e-3, strong_batch=128, strong_patience=10,
    # heads
    head_batch=256, head_epochs=80, head_patience=12, tune_head_epochs=40,
    q_layers=3,
    # end-to-end (item 1)
    e2e_seeds=[0, 1, 2], e2e_epochs=30, e2e_patience=6,
    e2e_variants_A=['bypass', 'quantum_v1', 'quantum_v2', 'classical_v2_matched'],
    e2e_variants_B=['bypass', 'quantum_v2', 'classical_v2_matched'],
    e2e_strong=False, e2e_variants_B_strong=['bypass', 'quantum_v2', 'classical_v2_matched'],
    # statistics
    delta=0.005, n_boot=2000,
    num_workers=2,
)
CFG_C = dict(n_block=5, q_layers=2, xi=0.8, hidden=128, num_steps=10, batch=128, epochs=15, patience=4,
             val_size=5000, seeds=list(range(10)), widths=[5, 20, 60], logit_widths=[5],
             run_xi_sweep=True, xi_grid=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0], sweep_seeds=[0, 1, 2])

if SMOKE_TEST:
    CFG.update(out='./qsnn_v3_smoke', eval_seeds=[0, 1], tune_seeds=[100], lr_grid=[1e-3, 1e-2],
               num_steps=3, enc_epochs=1, strong_steps=2, strong_epochs=1, head_epochs=2, head_patience=1,
               tune_head_epochs=1, e2e_seeds=[0], e2e_epochs=1, e2e_strong=True, n_boot=200, num_workers=0,
               prev_features_4='__none__', prev_features_16='__none__')
    CFG_C.update(seeds=[0, 1], epochs=1, patience=1, num_steps=3, widths=[5, 20], logit_widths=[5],
                 xi_grid=[0.0, 0.8], sweep_seeds=[0])

OUT = CFG['out']
if CFG['resume_from'] and os.path.isdir(CFG['resume_from']) and os.path.abspath(CFG['resume_from']) != os.path.abspath(OUT):
    shutil.copytree(CFG['resume_from'], OUT, dirs_exist_ok=True)
    print('resumed from', CFG['resume_from'])
for sub in ['', 'preds', 'labels', 'splits', 'checkpoints', 'features']:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def _ver(m):
    try:
        return __import__(m).__version__
    except Exception:
        return None

env = dict(python=sys.version.split()[0], platform=platform.platform(), torch=torch.__version__,
           cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None,
           torchvision=torchvision.__version__, snntorch=snn.__version__, numpy=np.__version__,
           scipy=_ver('scipy'), pandas=pd.__version__, qiskit=_ver('qiskit'), smoke_test=SMOKE_TEST,
           cfg=CFG, cfg_c=CFG_C)
json.dump(env, open(f'{OUT}/env.json', 'w'), indent=2, default=str)
open(f'{OUT}/requirements.txt', 'w').write('\n'.join(f'{k}=={v}' for k, v in
    [('torch', torch.__version__.split('+')[0]), ('torchvision', torchvision.__version__.split('+')[0]),
     ('snntorch', snn.__version__), ('numpy', np.__version__), ('scipy', env['scipy']),
     ('pandas', pd.__version__), ('qiskit', env['qiskit'])] if v) + '\n')
print(json.dumps({k: v for k, v in env.items() if k not in ('cfg', 'cfg_c')}, indent=1))

**Statistics and table generator** (the same file as the stand-alone `make_paper_tables.py`; written here so the notebook is self-contained).

In [ ]:
%%writefile make_paper_tables.py
#!/usr/bin/env python3
"""
make_paper_tables.py - build every table and figure for the QSNN paper from saved run results.

Usage:
    python make_paper_tables.py --results ./qsnn_v3_results [--delta 0.005] [--n-boot 2000]

Inputs (written by qsnn_v3.ipynb):
    <results>/runs_*.csv               one row per training run (stage = 'tune' or 'eval')
    <results>/preds/<exp>/*.npy        per-image test predictions for every eval run
    <results>/labels/<dataset>_test.npy
    <results>/circuit_diagnostics.json (optional)

Outputs (<results>/paper/):
    summary_<exp>.{csv,md,tex}          per-variant accuracy (mean, sd, 95% CI over seeds), params, chosen lr
    comparisons.{csv,md,tex}            every pre-registered comparison with all statistics and a verdict
    xi_sweep.{csv,md}                   Part C mixing-weight sweep with validation-selected xi (if run)
    circuit_diagnostics.md              effective-parameter counts and dead gates (if available)
    fig_<exp>.png                       bar charts

Statistics for each comparison (target vs control), over seeds present for both:
    paired t-test (Holm-corrected within each experiment), Wilcoxon signed-rank,
    TOST equivalence test with margin +-delta, paired bootstrap 95% CI of the accuracy difference
    over test images (seed-averaged correctness), and per-seed McNemar tests (median p).
Verdict:
    BETTER / WORSE   Holm p < alpha AND bootstrap CI excludes 0 (in that direction)
    EQUIVALENT       TOST p < alpha (difference within +-delta)
    INCONCLUSIVE     otherwise
"""
import argparse
import fnmatch
import glob
import json
import os

import numpy as np
import pandas as pd
from scipy import stats

# Pre-registered comparisons: (experiment pattern, target, controls). Holm family = one experiment.
COMPARISONS = [
    ('headsA', 'quantum_v1', ['bypass', 'classical_param', 'classical_width', 'fourier']),
    ('headsA', 'quantum_v2', ['quantum_v1', 'bypass', 'classical_width', 'classical_v2_matched']),
    ('headsA', 'quantum_v2_residual', ['bypass', 'classical_v2_matched_residual']),
    ('headsB', 'quantum_v2', ['bypass', 'classical_width', 'classical_v2_matched']),
    ('headsB', 'quantum_v2_residual', ['bypass', 'classical_v2_matched_residual']),
    ('headsB_strong', 'quantum_v2', ['bypass', 'classical_width', 'classical_v2_matched']),
    ('headsB_strong', 'quantum_v2_residual', ['bypass', 'classical_v2_matched_residual']),
    ('e2eA', 'quantum_v2', ['quantum_v1', 'bypass', 'classical_v2_matched']),
    ('e2eB', 'quantum_v2', ['bypass', 'classical_v2_matched']),
    ('e2eB', 'quantum_v2_residual', ['classical_v2_matched_residual']),
    ('e2eB_strong', 'quantum_v2', ['bypass', 'classical_v2_matched']),
    ('partC_w*', 'ppf_quantum', ['sfnn_only', 'ppf_linear', 'ppf_classical']),
]

EXPERIMENT_TITLES = {
    'headsA': 'CIFAR-10, 4 frozen features (base ConvSNN, rate coding)',
    'headsB': 'CIFAR-10, 16 frozen features (base ConvSNN, rate coding)',
    'headsB_strong': 'CIFAR-10, 16 frozen features (strong ConvSNN, direct coding, augmentation)',
    'e2eA': 'CIFAR-10, 4 features, end-to-end',
    'e2eB': 'CIFAR-10, 16 features, end-to-end',
    'e2eB_strong': 'CIFAR-10, 16 features, strong encoder, end-to-end',
}


# ----------------------------------------------------------------------------- loading
def load_runs(results):
    frames = [pd.read_csv(f) for f in sorted(glob.glob(os.path.join(results, 'runs_*.csv')))]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def eval_rows(runs, exp):
    d = runs[(runs.experiment == exp) & (runs.stage == 'eval')].copy()
    if exp.startswith('partC_w') and 'sfnn_only' not in set(d.variant):
        s = runs[(runs.experiment == 'partC_sfnn') & (runs.stage == 'eval')].copy()
        s['experiment'] = exp
        d = pd.concat([d, s], ignore_index=True)
    return d


_label_cache = {}


def correctness(results, rows):
    """{seed: bool array of per-image correctness} for the given eval rows."""
    out = {}
    for _, r in rows.iterrows():
        if not isinstance(r.get('pred_file'), str):
            continue
        p = os.path.join(results, r.pred_file)
        if not os.path.exists(p):
            continue
        ds = r.dataset
        if ds not in _label_cache:
            _label_cache[ds] = np.load(os.path.join(results, 'labels', f'{ds}_test.npy'))
        out[int(r.seed)] = np.load(p) == _label_cache[ds]
    return out


# ----------------------------------------------------------------------------- statistics
def holm(pvals):
    p = np.asarray(pvals, float)
    adj = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0][np.argsort(p[ok])]
    m, run = ok.sum(), 0.0
    for rank, i in enumerate(idx):
        run = max(run, min(1.0, (m - rank) * p[i]))
        adj[i] = run
    return adj


def tost_paired(d, delta):
    d = np.asarray(d, float)
    n, m = len(d), d.mean()
    if n < 2:
        return np.nan
    se = d.std(ddof=1) / np.sqrt(n)
    if se == 0:
        return 0.0 if abs(m) < delta else 1.0
    p_lower = 1 - stats.t.cdf((m + delta) / se, n - 1)
    p_upper = stats.t.cdf((m - delta) / se, n - 1)
    return float(max(p_lower, p_upper))


def bootstrap_ci(ca, cb, n_boot=2000, seed=0, chunk=200):
    """Paired bootstrap over test images of mean_seed(acc_a) - mean_seed(acc_b)."""
    diff = ca.mean(0) - cb.mean(0)
    N = len(diff)
    rng = np.random.default_rng(seed)
    boots = []
    for start in range(0, n_boot, chunk):
        b = min(chunk, n_boot - start)
        idx = rng.integers(0, N, size=(b, N), dtype=np.int32)
        boots.append(diff[idx].mean(1))
    boots = np.concatenate(boots)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def mcnemar_p(a, b):
    n01 = int((a & ~b).sum())
    n10 = int((~a & b).sum())
    if n01 + n10 == 0:
        return 1.0
    return float(stats.binomtest(n01, n01 + n10, 0.5).pvalue)


def compare(results, df, target, control, delta, n_boot):
    A, B = df[df.variant == target], df[df.variant == control]
    seeds = sorted(set(A.seed) & set(B.seed))
    row = dict(target=target, control=control, n_seeds=len(seeds))
    if len(seeds) == 0:
        return row
    a = A.set_index('seed').loc[seeds, 'test_acc'].values.astype(float)
    b = B.set_index('seed').loc[seeds, 'test_acc'].values.astype(float)
    d = a - b
    row.update(target_mean=a.mean(), control_mean=b.mean(), delta=d.mean())
    row['p_t'] = float(stats.ttest_rel(a, b).pvalue) if len(seeds) > 1 and d.std() > 0 else np.nan
    try:
        row['p_wilcoxon'] = float(stats.wilcoxon(a, b).pvalue) if len(seeds) > 1 and np.any(d != 0) else np.nan
    except ValueError:
        row['p_wilcoxon'] = np.nan
    row['p_tost'] = tost_paired(d, delta)
    cA, cB = correctness(results, A[A.seed.isin(seeds)]), correctness(results, B[B.seed.isin(seeds)])
    common = sorted(set(cA) & set(cB))
    if common:
        CA = np.stack([cA[s] for s in common]).astype(float)
        CB = np.stack([cB[s] for s in common]).astype(float)
        row['boot_lo'], row['boot_hi'] = bootstrap_ci(CA, CB, n_boot)
        mc = [mcnemar_p(cA[s], cB[s]) for s in common]
        row['mcnemar_median_p'] = float(np.median(mc))
        row['mcnemar_frac_sig'] = float(np.mean(np.array(mc) < 0.05))
    return row


def verdict(r, alpha=0.05):
    ph, lo, hi = r.get('p_holm', np.nan), r.get('boot_lo', np.nan), r.get('boot_hi', np.nan)
    if not np.isnan(ph) and ph < alpha:
        if r['delta'] > 0 and (np.isnan(lo) or lo > 0):
            return 'BETTER'
        if r['delta'] < 0 and (np.isnan(hi) or hi < 0):
            return 'WORSE'
    if not np.isnan(r.get('p_tost', np.nan)) and r['p_tost'] < alpha:
        return 'EQUIVALENT'
    return 'INCONCLUSIVE'


# ----------------------------------------------------------------------------- tables
def summary_table(df):
    agg = {'n': ('test_acc', 'size'), 'test_mean': ('test_acc', 'mean'), 'test_sd': ('test_acc', 'std'),
           'val_mean': ('val_acc', 'mean')}
    for c in ['lr', 'front_params', 'core_params', 'total_params']:
        if c in df and df[c].notna().any():
            agg[c] = (c, 'first')
    for c in ['train_acc', 'test_acc_snn', 'test_acc_second']:
        if c in df and df[c].notna().any():
            agg[c] = (c, 'mean')
    s = df.groupby('variant', sort=False).agg(**agg)
    s['ci95'] = [stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n) if n > 1 else np.nan for n, sd in zip(s.n, s.test_sd)]
    return s


def fmt(v, kind='acc'):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '-'
    if kind == 'acc':
        return f'{100 * v:.2f}'
    if kind == 'p':
        return '<0.001' if v < 1e-3 else f'{v:.3f}'
    if kind == 'int':
        return f'{int(v)}'
    if kind == 'lr':
        return f'{v:.0e}'
    return str(v)


def to_md(df, cols, kinds, headers):
    lines = ['| ' + ' | '.join(headers) + ' |', '|' + '---|' * len(headers)]
    for idx, r in df.iterrows():
        vals = [fmt(r[c] if c != '__index__' else idx, k) for c, k in zip(cols, kinds)]
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)


def to_tex(df, cols, kinds, headers, caption):
    esc = lambda s: str(s).replace('_', r'\_').replace('%', r'\%').replace('<', r'$<$').replace('±', r'$\pm$')
    out = [r'\begin{table}[t]', r'\centering', r'\small', r'\caption{' + esc(caption) + '}',
           r'\begin{tabular}{' + 'l' + 'r' * (len(headers) - 1) + '}', r'\toprule',
           ' & '.join(esc(h) for h in headers) + r' \\', r'\midrule']
    for idx, r in df.iterrows():
        out.append(' & '.join(esc(fmt(r[c] if c != '__index__' else idx, k)) for c, k in zip(cols, kinds)) + r' \\')
    out += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(out)


def write_summary(paper, exp, s):
    s = s.copy()
    s['acc'] = [f'{100 * m:.2f} ± {100 * sd:.2f}' if not np.isnan(sd) else f'{100 * m:.2f}' for m, sd in zip(s.test_mean, s.test_sd)]
    cols, kinds, heads = ['__index__', 'acc', 'ci95', 'n'], ['str', 'str', 'acc', 'int'], ['Variant', 'Test acc. (%)', '95% CI ±', 'Seeds']
    for c, k, h in [('lr', 'lr', 'LR'), ('front_params', 'int', 'Front params'), ('core_params', 'int', 'Core params'),
                    ('total_params', 'int', 'Total params'), ('test_acc_snn', 'acc', 'SNN branch'),
                    ('test_acc_second', 'acc', '2nd branch')]:
        if c in s:
            cols.append(c); kinds.append(k); heads.append(h)
    s.to_csv(os.path.join(paper, f'summary_{exp}.csv'))
    open(os.path.join(paper, f'summary_{exp}.md'), 'w').write(f'### {EXPERIMENT_TITLES.get(exp, exp)}\n\n' + to_md(s, cols, kinds, heads) + '\n')
    open(os.path.join(paper, f'summary_{exp}.tex'), 'w').write(to_tex(s, cols, kinds, heads, EXPERIMENT_TITLES.get(exp, exp)))


def write_figure(paper, exp, s):
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, ax = plt.subplots(figsize=(max(5, 1.1 * len(s)), 3.6))
    ax.bar(s.index, s.test_mean, yerr=s.test_sd, capsize=4,
           color=['#3b6fb6' if ('quantum' in v) else '#9aa5b1' for v in s.index])
    lo, hi = (s.test_mean - s.test_sd.fillna(0)).min(), (s.test_mean + s.test_sd.fillna(0)).max()
    ax.set_ylim(max(0, lo - 0.02), min(1, hi + 0.01))
    ax.set_ylabel('Test accuracy'); ax.set_title(EXPERIMENT_TITLES.get(exp, exp), fontsize=9)
    plt.xticks(rotation=25, ha='right', fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(paper, f'fig_{exp}.png'), dpi=200); plt.close(fig)


# ----------------------------------------------------------------------------- main
def main(results, delta=0.005, n_boot=2000, alpha=0.05, verbose=True):
    paper = os.path.join(results, 'paper')
    os.makedirs(paper, exist_ok=True)
    runs = load_runs(results)
    if runs.empty:
        print('no runs found in', results)
        return {}
    exps = sorted(e for e in runs.experiment.unique()
                  if not e.endswith('_tune') and e not in ('partC_sfnn', 'partC_xisweep'))
    out = {'summaries': {}}

    for exp in exps:
        df = eval_rows(runs, exp)
        if df.empty:
            continue
        s = summary_table(df)
        out['summaries'][exp] = s
        write_summary(paper, exp, s)
        write_figure(paper, exp, s)

    rows = []
    for pattern, target, controls in COMPARISONS:
        for exp in [e for e in exps if fnmatch.fnmatch(e, pattern)]:
            df = eval_rows(runs, exp)
            if target not in set(df.variant):
                continue
            for c in controls:
                if c in set(df.variant):
                    r = compare(results, df, target, c, delta, n_boot)
                    r['experiment'] = exp
                    rows.append(r)
    comp = pd.DataFrame(rows)
    if not comp.empty:
        comp['p_holm'] = np.nan
        for exp, idx in comp.groupby('experiment').groups.items():
            comp.loc[idx, 'p_holm'] = holm(comp.loc[idx, 'p_t'].values)
        comp['verdict'] = [verdict(r, alpha) for _, r in comp.iterrows()]
        comp = comp[['experiment', 'target', 'control', 'n_seeds', 'target_mean', 'control_mean', 'delta',
                     'boot_lo', 'boot_hi', 'p_t', 'p_holm', 'p_wilcoxon', 'p_tost', 'mcnemar_median_p',
                     'mcnemar_frac_sig', 'verdict']]
        comp.to_csv(os.path.join(paper, 'comparisons.csv'), index=False)
        c2 = comp.copy()
        c2['ci'] = [f'[{100 * lo:+.2f}, {100 * hi:+.2f}]' if not np.isnan(lo) else '-' for lo, hi in zip(c2.boot_lo, c2.boot_hi)]
        c2['d'] = [f'{100 * d:+.2f}' for d in c2.delta]
        c2['pair'] = c2.target + ' vs ' + c2.control
        c2 = c2.set_index('experiment')
        cols = ['__index__', 'pair', 'n_seeds', 'd', 'ci', 'p_holm', 'p_wilcoxon', 'p_tost', 'mcnemar_median_p', 'verdict']
        kinds = ['str', 'str', 'int', 'str', 'str', 'p', 'p', 'p', 'p', 'str']
        heads = ['Experiment', 'Comparison', 'Seeds', 'Δ (pts)', 'Bootstrap 95% CI', 'Holm p', 'Wilcoxon p',
                 f'TOST p (±{100 * delta:.1f})', 'McNemar p (median)', 'Verdict']
        open(os.path.join(paper, 'comparisons.md'), 'w').write(to_md(c2, cols, kinds, heads) + '\n')
        open(os.path.join(paper, 'comparisons.tex'), 'w').write(
            to_tex(c2, cols, kinds, heads, f'Paired comparisons (Holm within experiment; equivalence margin ±{100 * delta:.1f} pts)'))
    out['comparisons'] = comp

    sw = runs[(runs.experiment == 'partC_xisweep') & (runs.stage == 'eval')]
    if not sw.empty:
        g = sw.groupby(['variant', 'xi']).agg(val=('val_acc', 'mean'), test=('test_acc', 'mean'), n=('seed', 'size')).reset_index()
        g.to_csv(os.path.join(paper, 'xi_sweep.csv'), index=False)
        sel = g.loc[g.groupby('variant').val.idxmax()]
        md = ['| Variant | ξ | Val acc | Test acc | Seeds |', '|---|---|---|---|---|']
        md += [f'| {r.variant} | {r.xi:.1f} | {100 * r.val:.2f} | {100 * r.test:.2f} | {r.n} |' for _, r in g.iterrows()]
        md += ['', 'Validation-selected ξ:', ''] + [f'- {r.variant}: ξ = {r.xi:.1f} → test {100 * r.test:.2f}%' for _, r in sel.iterrows()]
        open(os.path.join(paper, 'xi_sweep.md'), 'w').write('\n'.join(md) + '\n')
        out['xi_sweep'] = g

    diag_path = os.path.join(results, 'circuit_diagnostics.json')
    if os.path.exists(diag_path):
        diag = json.load(open(diag_path))
        md = ['| Circuit | Effective / total params | Zero-gradient gates | Median input σ_min |', '|---|---|---|---|']
        for name, d in diag.items():
            md.append(f"| {name} | {d.get('effective', '-')}/{d.get('total', '-')} | {len(d.get('dead', []))} | "
                      f"{d['sigma_min_median']:.3f} |" if 'sigma_min_median' in d else
                      f"| {name} | {d.get('effective', '-')}/{d.get('total', '-')} | {len(d.get('dead', []))} | - |")
        open(os.path.join(paper, 'circuit_diagnostics.md'), 'w').write('\n'.join(md) + '\n')

    if verbose:
        for exp, s in out['summaries'].items():
            print(f'\n== {exp}: {EXPERIMENT_TITLES.get(exp, exp)}')
            print((s[['n', 'test_mean', 'test_sd', 'ci95']] * [1, 100, 100, 100]).round(2).to_string())
        if not comp.empty:
            print('\n== comparisons')
            print(comp[['experiment', 'target', 'control', 'n_seeds', 'delta', 'p_holm', 'p_tost', 'verdict']]
                  .assign(delta=lambda x: (100 * x.delta).round(2)).to_string(index=False))
        print(f'\nTables and figures written to {paper}')
    return out


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('--results', default='./qsnn_v3_results')
    ap.add_argument('--delta', type=float, default=0.005, help='equivalence margin (accuracy fraction), default 0.005 = 0.5 pts')
    ap.add_argument('--n-boot', type=int, default=2000)
    ap.add_argument('--alpha', type=float, default=0.05)
    a = ap.parse_args()
    main(a.results, a.delta, a.n_boot, a.alpha)

In [ ]:
import importlib, make_paper_tables as mpt
importlib.reload(mpt)
print('make_paper_tables loaded:', len(mpt.COMPARISONS), 'comparison groups')

## Stage 1 — data

In [ ]:
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

def _split(n, val_size, name):
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(1234))
    val_idx, tr_idx = perm[:val_size], perm[val_size:]
    np.save(f'{OUT}/splits/{name}_val_idx.npy', val_idx.numpy()); np.save(f'{OUT}/splits/{name}_train_idx.npy', tr_idx.numpy())
    return tr_idx.tolist(), val_idx.tolist()

def labels_of(ds):
    if isinstance(ds, TensorDataset):
        return ds.tensors[1].numpy()
    return np.asarray(ds.targets)

def save_labels(name, test_ds):
    np.save(f'{OUT}/labels/{name}_test.npy', labels_of(test_ds).astype(np.int64))

def _fake(n, shape, seed):
    g = torch.Generator().manual_seed(seed)
    return TensorDataset(torch.rand(n, *shape, generator=g), torch.randint(0, 10, (n,), generator=g))

def load_cifar(kind):
    # kind='base': [0,1] pixels (rate coding). kind='strong': normalised; train split augmented.
    if SMOKE_TEST:
        tr, va, te = _fake(512, (3, 32, 32), 0), _fake(128, (3, 32, 32), 1), _fake(256, (3, 32, 32), 2)
        return dict(train=tr, train_eval=tr, val=va, test=te)
    if kind == 'base':
        tf_tr = tf_ev = T.ToTensor()
    else:
        tf_ev = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
        tf_tr = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    full_tr = torchvision.datasets.CIFAR10(CFG['data_root'], train=True, download=True, transform=tf_tr)
    full_ev = torchvision.datasets.CIFAR10(CFG['data_root'], train=True, download=True, transform=tf_ev)
    test = torchvision.datasets.CIFAR10(CFG['data_root'], train=False, download=True, transform=tf_ev)
    tr_idx, val_idx = _split(len(full_tr), 5000, 'cifar10')     # same seeded split as v1/v2
    return dict(train=Subset(full_tr, tr_idx), train_eval=Subset(full_ev, tr_idx), val=Subset(full_ev, val_idx), test=test)

def load_mnist():
    if SMOKE_TEST:
        tr, va, te = _fake(512, (1, 28, 28), 3), _fake(128, (1, 28, 28), 4), _fake(256, (1, 28, 28), 5)
        return dict(train=tr, val=va, test=te)
    tf = T.ToTensor()
    full = torchvision.datasets.MNIST(CFG['data_root'], train=True, download=True, transform=tf)
    test = torchvision.datasets.MNIST(CFG['data_root'], train=False, download=True, transform=tf)
    tr_idx, val_idx = _split(len(full), CFG_C['val_size'], 'mnist')
    return dict(train=Subset(full, tr_idx), val=Subset(full, val_idx), test=test)

need_cifar = CFG['run_heads'] or CFG['run_strong'] or CFG['run_e2e']
cifar_base = load_cifar('base') if need_cifar else None
cifar_strong = load_cifar('strong') if (CFG['run_strong'] or (CFG['run_e2e'] and CFG['e2e_strong'])) else None
mnist = load_mnist() if CFG['run_partC'] else None
if cifar_base: save_labels('cifar10', cifar_base['test']); print('CIFAR-10', {k: len(v) for k, v in cifar_base.items()})
if mnist: save_labels('mnist', mnist['test']); print('MNIST', {k: len(v) for k, v in mnist.items()})

def rate_encode(x, num_steps, generator=None):
    r = torch.rand((num_steps,) + tuple(x.shape), device=x.device, generator=generator)
    return (r < x.unsqueeze(0)).float()

## Stage 1b — exact state-vector simulator and all circuits

In [ ]:
def _cdt(t):
    return torch.complex128 if t.dtype == torch.float64 else torch.complex64

def init_state(B, n, dtype, device):
    psi = torch.zeros((B,) + (2,) * n, dtype=dtype, device=device)
    psi.view(B, -1)[:, 0] = 1
    return psi

def ry_mat(t):
    c, s = torch.cos(t / 2).to(_cdt(t)), torch.sin(t / 2).to(_cdt(t))
    return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)

def rx_mat(t):
    c, s = torch.cos(t / 2).to(_cdt(t)), torch.sin(t / 2).to(_cdt(t))
    return torch.stack([torch.stack([c, -1j * s], -1), torch.stack([-1j * s, c], -1)], -2)

def rz_mat(t):
    e0, e1 = torch.exp(-0.5j * t.to(_cdt(t))), torch.exp(0.5j * t.to(_cdt(t)))
    z = torch.zeros_like(e0)
    return torch.stack([torch.stack([e0, z], -1), torch.stack([z, e1], -1)], -2)

_H = torch.tensor([[1.0, 1.0], [1.0, -1.0]], dtype=torch.float64) / math.sqrt(2)
_ZZ = torch.tensor([[1.0, -1.0], [-1.0, 1.0]], dtype=torch.float64)

def apply_1q(psi, U, q):
    p = psi.movedim(q + 1, -1)
    return torch.einsum('b...j,bij->b...i', p, U).movedim(-1, q + 1)

def apply_cx(psi, c, t):
    p = psi.movedim((c + 1, t + 1), (1, 2))
    p = torch.stack([p[:, 0], p[:, 1].flip(1)], dim=1)
    return p.movedim((1, 2), (c + 1, t + 1))

def apply_rzz(psi, i, j, theta):
    p = psi.movedim((i + 1, j + 1), (1, 2))
    phase = torch.exp(-0.5j * theta.to(_cdt(theta)) * _ZZ.to(theta.device).to(_cdt(theta)))
    p = p * phase.view(1, 2, 2, *([1] * (p.dim() - 3)))
    return p.movedim((1, 2), (i + 1, j + 1))

def probabilities(psi):
    n = psi.dim() - 1
    return (psi.abs() ** 2).permute(0, *range(n, 0, -1)).reshape(psi.shape[0], -1)

def z_readout(psi):
    B, n = psi.shape[0], psi.dim() - 1
    return torch.stack([(lambda a: (a[:, 0].abs() ** 2 - a[:, 1].abs() ** 2).sum(-1))(psi.movedim(i + 1, 1).reshape(B, 2, -1))
                        for i in range(n)], 1)

def pauli_readout(psi):
    B, n = psi.shape[0], psi.dim() - 1
    X, Y, Z, ZZ = [], [], [], []
    for i in range(n):
        a = psi.movedim(i + 1, 1).reshape(B, 2, -1)
        c = (a[:, 0].conj() * a[:, 1]).sum(-1)
        X.append(2 * c.real); Y.append(2 * c.imag)
        Z.append((a[:, 0].abs() ** 2 - a[:, 1].abs() ** 2).sum(-1))
    for i in range(n):
        j = (i + 1) % n
        p = (psi.movedim((i + 1, j + 1), (1, 2)).abs() ** 2).reshape(B, 2, 2, -1).sum(-1)
        ZZ.append(p[:, 0, 0] - p[:, 0, 1] - p[:, 1, 0] + p[:, 1, 1])
    return torch.stack(X + Y + Z + ZZ, dim=1)


def original_probs(x, w):
    # circuit of the original model.py: Ry(x) -> Rx,Rz,Ry,Rz(w) -> CNOT(0,1), CNOT(2,3)
    B = x.shape[0]
    psi = init_state(B, 4, _cdt(x), x.device)
    for i in range(4): psi = apply_1q(psi, ry_mat(x[:, i]), i)
    for i, (f, k) in enumerate([(rx_mat, 0), (rz_mat, 1), (ry_mat, 2), (rz_mat, 3)]):
        psi = apply_1q(psi, f(w[k].expand(B)), i)
    psi = apply_cx(psi, 0, 1); psi = apply_cx(psi, 2, 3)
    return probabilities(psi)

def v1_probs(x, w, L=2):
    B, n = x.shape
    psi = init_state(B, n, _cdt(x), x.device); k = 0
    for _ in range(L):
        for i in range(n): psi = apply_1q(psi, ry_mat(x[:, i]), i)
        for i in range(n):
            psi = apply_1q(psi, rz_mat(w[k].expand(B)), i); k += 1
            psi = apply_1q(psi, ry_mat(w[k].expand(B)), i); k += 1
        for i in range(n): psi = apply_cx(psi, i, (i + 1) % n)
    for i in range(n):
        psi = apply_1q(psi, ry_mat(w[k].expand(B)), i); k += 1
    return probabilities(psi)

def v2_forward(x, scale, bias, rot, ent):
    B, n = x.shape
    psi = init_state(B, n, _cdt(x), x.device)
    for l in range(scale.shape[0]):
        for i in range(n): psi = apply_1q(psi, ry_mat(scale[l, i] * x[:, i] + bias[l, i]), i)
        for i in range(n):
            psi = apply_1q(psi, rz_mat(rot[l, i, 0].expand(B)), i)
            psi = apply_1q(psi, ry_mat(rot[l, i, 1].expand(B)), i)
        for i in range(n): psi = apply_rzz(psi, i, (i + 1) % n, ent[l, i])
    return pauli_readout(psi)

def v2_init(n=4, L=3, noise=0.05, seed=0, dtype=torch.float32):
    g = torch.Generator().manual_seed(seed)
    r = lambda *s: noise * torch.randn(*s, generator=g, dtype=dtype)
    return dict(scale=torch.full((L, n), 1.0 / L, dtype=dtype) + r(L, n), bias=r(L, n), rot=r(L, n, 2), ent=r(L, n))

def v2_param_count(n=4, L=3):
    return 5 * L * n

def xu_circuit(x, w):
    # Xu et al.: H, RY(x) -> [CNOT chain -> RZ RX RZ] x L -> <Z_i>.   w: (L, n, 3)
    B, n = x.shape
    psi = init_state(B, n, _cdt(x), x.device)
    Hm = _H.to(x.device).to(_cdt(x)).expand(B, 2, 2)
    for i in range(n):
        psi = apply_1q(psi, Hm, i); psi = apply_1q(psi, ry_mat(x[:, i]), i)
    for l in range(w.shape[0]):
        for i in range(n - 1): psi = apply_cx(psi, i, i + 1)
        for i in range(n):
            psi = apply_1q(psi, rz_mat(w[l, i, 0].expand(B)), i)
            psi = apply_1q(psi, rx_mat(w[l, i, 1].expand(B)), i)
            psi = apply_1q(psi, rz_mat(w[l, i, 2].expand(B)), i)
    return z_readout(psi)

## Stage 1c — circuit diagnostics (no data needed)
Qiskit cross-check of every circuit; effective-parameter count (rank of the Jacobian with respect to the weights); zero-gradient gates; input sensitivity (median smallest singular value of the input Jacobian; ≈ 0 means the circuit loses information).

In [ ]:
def jacobian_rank(fn, n_params, n_in, n_x=64, n_w=5, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = ((torch.rand(n_x, n_in, generator=g) * 2 - 1) * math.pi).double()
    ranks, coln = [], torch.zeros(n_params, dtype=torch.float64)
    for _ in range(n_w):
        w = ((torch.rand(n_params, generator=g) * 2 - 1) * math.pi).double()
        J = torch.autograd.functional.jacobian(lambda w_: fn(x, w_).reshape(-1), w)
        s = torch.linalg.svdvals(J); ranks.append(int((s > 1e-8 * s.max()).sum())); coln += J.norm(dim=0) / n_w
    return max(ranks), [int(i) for i in torch.where(coln < 1e-10)[0]]

def input_sigma_min(fn, n_in, n=1000, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = ((torch.rand(n, n_in, generator=g) * 2 - 1) * math.pi).double()
    J = torch.func.vmap(torch.func.jacrev(lambda xi: fn(xi[None])[0]))(x)
    return torch.linalg.svdvals(J)[:, -1]

if CFG['run_diagnostics']:
    L2, LX, nX = CFG['q_layers'], CFG_C['q_layers'], CFG_C['n_block']
    try:
        from qiskit import QuantumCircuit
        from qiskit.quantum_info import Statevector, SparsePauliOp
        rng = np.random.default_rng(0)
        def qk_probs(qc): return Statevector.from_instruction(qc).probabilities()
        def qk_exp(qc, terms, n): return np.array([Statevector.from_instruction(qc).expectation_value(SparsePauliOp.from_sparse_list(t, n)).real for t in terms])
        errs = {}
        # original
        xv, wv = rng.uniform(-np.pi, np.pi, 4), rng.uniform(-np.pi, np.pi, 4)
        qc = QuantumCircuit(4); [qc.ry(xv[i], i) for i in range(4)]
        qc.rx(wv[0], 0); qc.rz(wv[1], 1); qc.ry(wv[2], 2); qc.rz(wv[3], 3); qc.cx(0, 1); qc.cx(2, 3)
        errs['original'] = np.abs(qk_probs(qc) - original_probs(torch.tensor(xv[None]), torch.tensor(wv))[0].numpy()).max()
        # v1
        xv, wv = rng.uniform(-np.pi, np.pi, 4), rng.uniform(-np.pi, np.pi, 20)
        qc, k = QuantumCircuit(4), 0
        for _ in range(2):
            for i in range(4): qc.ry(xv[i], i)
            for i in range(4): qc.rz(wv[k], i); qc.ry(wv[k + 1], i); k += 2
            for i in range(4): qc.cx(i, (i + 1) % 4)
        for i in range(4): qc.ry(wv[k], i); k += 1
        errs['v1'] = np.abs(qk_probs(qc) - v1_probs(torch.tensor(xv[None]), torch.tensor(wv))[0].numpy()).max()
        # v2
        P = {k_: v.double() for k_, v in v2_init(4, L2, noise=1.0, seed=1, dtype=torch.float64).items()}
        xv = rng.uniform(-np.pi, np.pi, 4); qc = QuantumCircuit(4)
        for l in range(L2):
            for i in range(4): qc.ry(float(P['scale'][l, i] * xv[i] + P['bias'][l, i]), i)
            for i in range(4): qc.rz(float(P['rot'][l, i, 0]), i); qc.ry(float(P['rot'][l, i, 1]), i)
            for i in range(4): qc.rzz(float(P['ent'][l, i]), i, (i + 1) % 4)
        terms = [[(b, [i], 1.0)] for b in 'XYZ' for i in range(4)] + [[('ZZ', [i, (i + 1) % 4], 1.0)] for i in range(4)]
        errs['v2'] = np.abs(qk_exp(qc, terms, 4) - v2_forward(torch.tensor(xv[None]), **P)[0].numpy()).max()
        # Xu
        xv, wv = rng.uniform(-np.pi, np.pi, nX), rng.uniform(-np.pi, np.pi, (LX, nX, 3))
        qc = QuantumCircuit(nX)
        for i in range(nX): qc.h(i); qc.ry(xv[i], i)
        for l in range(LX):
            for i in range(nX - 1): qc.cx(i, i + 1)
            for i in range(nX): qc.rz(wv[l, i, 0], i); qc.rx(wv[l, i, 1], i); qc.rz(wv[l, i, 2], i)
        errs['xu'] = np.abs(qk_exp(qc, [[('Z', [i], 1.0)] for i in range(nX)], nX) - xu_circuit(torch.tensor(xv[None]), torch.tensor(wv))[0].numpy()).max()
        for k_, e in errs.items():
            print(f'qiskit cross-check {k_:9s} max err {e:.2e}', 'OK' if e < 1e-10 else 'MISMATCH')
            assert e < 1e-10
    except ImportError:
        print('Qiskit not installed - cross-checks skipped.')

    nv2 = v2_param_count(4, L2)
    unpack_v2 = lambda w: dict(scale=w[:L2 * 4].view(L2, 4), bias=w[L2 * 4:2 * L2 * 4].view(L2, 4),
                               rot=w[2 * L2 * 4:4 * L2 * 4].view(L2, 4, 2), ent=w[4 * L2 * 4:].view(L2, 4))
    circuits = {
        'original (model.py)': (lambda x, w: original_probs(x, w), 4, 4),
        'v1 re-uploading': (lambda x, w: v1_probs(x, w), 20, 4),
        'v2 (this work)': (lambda x, w: v2_forward(x, **unpack_v2(w)), nv2, 4),
        'Xu et al. PPF-SQNN': (lambda x, w: xu_circuit(x, w.view(LX, nX, 3)), 3 * LX * nX, nX),
    }
    diag = {}
    for name, (fn, npar, nin) in circuits.items():
        eff, dead = jacobian_rank(fn, npar, nin)
        diag[name] = dict(effective=eff, total=npar, dead=dead)
    g = torch.Generator().manual_seed(0)
    w1 = ((torch.rand(20, generator=g) * 2 - 1) * math.pi).double()
    P0 = {k_: v.double() for k_, v in v2_init(4, L2, seed=0, dtype=torch.float64).items()}
    diag['v1 re-uploading']['sigma_min_median'] = float(input_sigma_min(lambda x: v1_probs(x, w1), 4).median())
    diag['v2 (this work)']['sigma_min_median'] = float(input_sigma_min(lambda x: v2_forward(x, **P0), 4).median())
    json.dump(diag, open(f'{OUT}/circuit_diagnostics.json', 'w'), indent=2)
    for k_, d in diag.items():
        print(f"{k_:22s} effective {d['effective']:3d}/{d['total']:3d}  dead gates {len(d['dead'])}"
              + (f"  median sigma_min {d['sigma_min_median']:.3f}" if 'sigma_min_median' in d else ''))

## Stage 1d — models

In [ ]:
class ConvSNN(nn.Module):
    # base encoder (v1/v2): rate-coded input, 2 conv + 3 FC LIF layers
    def __init__(self, beta=0.9, n_out=4):
        super().__init__()
        sg = surrogate.fast_sigmoid()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1); self.lif1 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True); self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.lif2 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True); self.pool2 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128); self.lif3 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True)
        self.fc2 = nn.Linear(128, 64); self.lif4 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True)
        self.fc3 = nn.Linear(64, n_out); self.lif5 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True, output=True)

    def forward(self, inp):
        snn_utils.reset(self)
        rec = []
        for t in range(inp.shape[0]):
            x = self.pool1(self.lif1(self.conv1(inp[t])))
            x = self.pool2(self.lif2(self.conv2(x)))
            x = self.lif4(self.fc2(self.lif3(self.fc1(x.flatten(1)))))
            _, mem = self.lif5(self.fc3(x)); rec.append(mem)
        return torch.stack(rec).mean(0)


class ConvSNNStrong(nn.Module):
    # stronger encoder: 4 conv layers with BatchNorm, direct (analog) input current
    def __init__(self, beta=0.9, n_out=16):
        super().__init__()
        sg = surrogate.fast_sigmoid()
        chans = [(3, 64), (64, 128), (128, 256), (256, 256)]
        self.convs = nn.ModuleList([nn.Conv2d(a, b, 3, padding=1, bias=False) for a, b in chans])
        self.bns = nn.ModuleList([nn.BatchNorm2d(b) for _, b in chans])
        self.lifs = nn.ModuleList([snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True) for _ in chans])
        self.pool_after = {1, 2, 3}                                  # 32 -> 16 -> 8 -> 4
        self.fc1 = nn.Linear(256 * 4 * 4, 256); self.lif_fc = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True)
        self.fc2 = nn.Linear(256, n_out); self.lif_out = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=True, output=True)

    def forward(self, inp):
        snn_utils.reset(self)
        rec = []
        for t in range(inp.shape[0]):
            x = inp[t]
            for k, (cv, bn, lif) in enumerate(zip(self.convs, self.bns, self.lifs)):
                x = lif(bn(cv(x)))
                if k in self.pool_after: x = F.max_pool2d(x, 2)
            x = self.lif_fc(self.fc1(x.flatten(1)))
            _, mem = self.lif_out(self.fc2(x)); rec.append(mem)
        return torch.stack(rec).mean(0)


class ClassicalANN(nn.Module):
    def __init__(self, input_dim, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.Sigmoid(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.Sigmoid(), nn.Dropout(dropout),
            nn.Linear(32, num_classes))

    def forward(self, x):
        return self.net(x)


# ---------------- front-ends: bounded angles (B, n_feat) -> features ----------------
class Identity(nn.Module):
    def __init__(self, n): super().__init__(); self.out_dim = n
    def forward(self, a): return a

class QuantumV1(nn.Module):
    def __init__(self, seed=0):
        super().__init__()
        self.w = nn.Parameter((torch.rand(20, generator=torch.Generator().manual_seed(seed)) * 2 - 1) * math.pi); self.out_dim = 16
    def forward(self, a): return v1_probs(a, self.w)

class ClassicalParamMatched(nn.Module):       # 20 params, matched to v1 (as in notebook v1)
    def __init__(self, seed=0):
        super().__init__()
        self.lin = nn.Linear(4, 4); self.register_buffer('proj', torch.randn(4, 16, generator=torch.Generator().manual_seed(seed))); self.out_dim = 16
    def forward(self, a): return F.softmax(torch.tanh(self.lin(a)) @ self.proj, -1)

class FourierSurrogate(nn.Module):             # random Fourier features on v1's integer spectrum
    def __init__(self, L=2, D=16, seed=0):
        super().__init__()
        freqs = [f for f in itertools.product(range(-L, L + 1), repeat=4) if any(f) and f > tuple(-v for v in f)]
        idx = np.random.default_rng(seed).choice(len(freqs), size=D, replace=False)
        self.register_buffer('omega', torch.tensor([freqs[i] for i in idx], dtype=torch.float32))
        self.lin = nn.Linear(2 * D, 16); self.out_dim = 16
    def forward(self, a):
        z = a @ self.omega.T
        return F.softmax(self.lin(torch.cat([torch.cos(z), torch.sin(z)], -1)), -1)

class ClassicalWidth(nn.Module):
    def __init__(self, n_in, n_out): super().__init__(); self.lin = nn.Linear(n_in, n_out); self.out_dim = n_out
    def forward(self, a): return F.softmax(self.lin(a), -1)

class QuantumV2(nn.Module):
    def __init__(self, G=1, L=3, seed=0):
        super().__init__()
        inits = [v2_init(4, L, seed=seed * 100 + g) for g in range(G)]
        for k in ['scale', 'bias', 'rot', 'ent']:
            setattr(self, k, nn.Parameter(torch.stack([d[k] for d in inits])))
        self.G, self.out_dim = G, 16 * G
    def forward(self, a):
        return torch.cat([v2_forward(a[:, 4 * g:4 * g + 4], self.scale[g], self.bias[g], self.rot[g], self.ent[g]) for g in range(self.G)], 1)

class ClassicalV2Matched(nn.Module):
    def __init__(self, G=1, n_params=60, seed=0):
        super().__init__()
        h = n_params // 5
        self.lins = nn.ModuleList([nn.Linear(4, h) for _ in range(G)])
        self.register_buffer('proj', torch.randn(G, h, 16, generator=torch.Generator().manual_seed(seed)) / math.sqrt(h))
        self.G, self.out_dim = G, 16 * G
    def forward(self, a):
        return torch.cat([torch.tanh(torch.tanh(self.lins[g](a[:, 4 * g:4 * g + 4])) @ self.proj[g]) for g in range(self.G)], 1)

class Residual(nn.Module):
    def __init__(self, front, n_in): super().__init__(); self.front, self.out_dim = front, front.out_dim + n_in
    def forward(self, a): return torch.cat([self.front(a), a], 1)

def make_frontend(variant, n_feat, seed):
    G, L = n_feat // 4, CFG['q_layers']
    base = variant.replace('_residual', '')
    f = {'bypass': lambda: Identity(n_feat),
         'classical_width': lambda: ClassicalWidth(n_feat, 4 * n_feat),
         'classical_param': lambda: ClassicalParamMatched(seed),
         'fourier': lambda: FourierSurrogate(2, 16, seed),
         'quantum_v1': lambda: QuantumV1(seed),
         'quantum_v2': lambda: QuantumV2(G, L, seed),
         'classical_v2_matched': lambda: ClassicalV2Matched(G, v2_param_count(4, L), seed)}[base]()
    return Residual(f, n_feat) if variant.endswith('_residual') else f

class HeadModel(nn.Module):
    def __init__(self, variant, n_feat, seed):
        super().__init__()
        self.front = make_frontend(variant, n_feat, seed)
        self.ann = ClassicalANN(self.front.out_dim, 10, CFG['dropout'])
    def forward(self, angles): return self.ann(self.front(angles))

class FullModel(nn.Module):
    def __init__(self, variant, n_feat, seed, enc='base'):
        super().__init__()
        set_seed(seed)
        self.kind = enc
        self.enc = ConvSNN(CFG['beta'], n_feat) if enc == 'base' else ConvSNNStrong(CFG['beta'], n_feat)
        self.head = HeadModel(variant, n_feat, seed)
    def encode(self, x, gen=None):
        if self.kind == 'base':
            return rate_encode(x, CFG['num_steps'], gen)
        return x.unsqueeze(0).expand(CFG['strong_steps'], *x.shape)     # direct input current
    def angles(self, x, gen=None): return math.pi * torch.tanh(self.enc(self.encode(x, gen)))
    def forward(self, x, gen=None): return self.head(self.angles(x, gen))

def n_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

VARIANTS_A = ['bypass', 'classical_width', 'classical_param', 'fourier', 'quantum_v1', 'quantum_v2',
              'classical_v2_matched', 'quantum_v2_residual', 'classical_v2_matched_residual']
VARIANTS_B = ['bypass', 'classical_width', 'quantum_v2', 'classical_v2_matched',
              'quantum_v2_residual', 'classical_v2_matched_residual']
for name, nf, vs in [('A', 4, VARIANTS_A), ('B', 16, VARIANTS_B)]:
    print(f'Part {name} ({nf} features)')
    for v in vs:
        h = HeadModel(v, nf, 0)
        print(f'  {v:30s} front {n_trainable(h.front):5d} | out {h.front.out_dim:3d} | head {n_trainable(h):6d}')

## Stage 1e — run registry (resumable), training loops, learning-rate tuning

In [ ]:
ALL_COLS = ['key', 'experiment', 'stage', 'variant', 'seed', 'lr', 'xi', 'width', 'fusion', 'val_acc', 'test_acc',
            'train_acc', 'val_acc_snn', 'val_acc_second', 'test_acc_snn', 'test_acc_second', 'epochs', 'seconds',
            'front_params', 'core_params', 'total_params', 'dataset', 'pred_file']

def runs_file(exp): return f'{OUT}/runs_{exp}.csv'

def make_key(stage, variant, seed, lr, extra=''):
    return f'{stage}|{variant}|{seed}|{lr:.0e}|{extra}'

def done_keys(exp):
    f = runs_file(exp)
    return set(pd.read_csv(f).key) if os.path.exists(f) else set()

def record(exp, row, preds=None):
    row = dict(row, experiment=exp)
    if preds is not None:
        rel = f"preds/{exp}/{row['variant']}__{row.get('extra_tag', '')}lr{row['lr']:.0e}__s{row['seed']}.npy"
        os.makedirs(os.path.dirname(f'{OUT}/{rel}'), exist_ok=True)
        np.save(f'{OUT}/{rel}', np.asarray(preds, dtype=np.int8)); row['pred_file'] = rel
    df = pd.DataFrame([{c: row.get(c) for c in ALL_COLS}])
    f = runs_file(exp)
    df.to_csv(f, mode='a', header=not os.path.exists(f), index=False)

def lr_choices():
    f = f'{OUT}/lr_choice.json'
    return json.load(open(f)) if os.path.exists(f) else {}

def save_lr_choice(exp, variant, lr):
    d = lr_choices(); d.setdefault(exp, {})[variant] = lr
    json.dump(d, open(f'{OUT}/lr_choice.json', 'w'), indent=2)

def choose_lr(exp, variant, run_one, extra='', tune_exp=None):
    # run_one(variant, seed, lr, stage) -> (row, preds). Tuning rows live in <exp>_tune.
    tune_exp = tune_exp or f'{exp}_tune'
    if not CFG['tune_lr']:
        return CFG['default_lr']
    done = done_keys(tune_exp)
    for lr in CFG['lr_grid']:
        for s in CFG['tune_seeds']:
            k = make_key('tune', variant, s, lr, extra)
            if k in done: continue
            row, _ = run_one(variant, s, lr, 'tune')
            record(tune_exp, dict(row, key=k, stage='tune'))
    t = pd.read_csv(runs_file(tune_exp))
    t = t[(t.variant == variant) & (t.key.str.endswith('|' + extra))]
    best = float(t.groupby('lr').val_acc.mean().idxmax())
    save_lr_choice(exp, variant, best)
    return best

def run_experiment(exp, variants, run_one, seeds, extra='', tune_exp=None, lr_from=None):
    for v in variants:
        if lr_from is not None:
            lr = lr_from(v)
        else:
            lr = choose_lr(exp, v, run_one, extra, tune_exp)
        done = done_keys(exp)
        for s in seeds:
            k = make_key('eval', v, s, lr, extra)
            if k in done: continue
            t0 = time.time()
            row, preds = run_one(v, s, lr, 'eval')
            record(exp, dict(row, key=k, stage='eval'), preds)
            print(f"[{exp}] {v:30s} lr {lr:.0e} seed {s}  val {row['val_acc']:.4f}  test {row['test_acc']:.4f}  ({time.time()-t0:.0f}s)")


# ---------------- head training on frozen features ----------------
@torch.no_grad()
def head_predict(model, A, bs=4096):
    model.eval()
    return torch.cat([model(A[i:i + bs].to(DEVICE)).argmax(1).cpu() for i in range(0, len(A), bs)])

def train_head(variant, n_feat, seed, feats, lr, max_epochs):
    set_seed(seed)
    model = HeadModel(variant, n_feat, seed).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    (Atr, Ytr), (Ava, Yva), (Ate, Yte) = feats['train'], feats['val'], feats['test']
    g = torch.Generator().manual_seed(seed)
    best, best_state, bad, ep, t0 = -1.0, None, 0, 0, time.time()
    for ep in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(len(Atr), generator=g)
        for i in range(0, len(perm), CFG['head_batch']):
            idx = perm[i:i + CFG['head_batch']]
            if len(idx) < 2: continue
            loss = F.cross_entropy(model(Atr[idx].to(DEVICE)), Ytr[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        vacc = (head_predict(model, Ava) == Yva).float().mean().item()
        if vacc > best: best, best_state, bad = vacc, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= CFG['head_patience']: break
    model.load_state_dict(best_state)
    pred = head_predict(model, Ate)
    return dict(variant=variant, seed=seed, lr=lr, val_acc=best, test_acc=(pred == Yte).float().mean().item(), epochs=ep,
                seconds=round(time.time() - t0, 1), front_params=n_trainable(model.front), total_params=n_trainable(model),
                dataset='cifar10'), pred.numpy()

def head_runner(n_feat, feats):
    def run_one(v, s, lr, stage):
        return train_head(v, n_feat, s, feats, lr, CFG['tune_head_epochs'] if stage == 'tune' else CFG['head_epochs'])
    return run_one


# ---------------- full (encoder + head) training ----------------
def full_epoch(model, loader, opt=None, gen=None, sched=None):
    train = opt is not None
    model.train(train); n = correct = 0; preds = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            out = model(x, gen); loss = F.cross_entropy(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        p = out.argmax(1); n += y.numel(); correct += (p == y).sum().item()
        if not train: preds.append(p.cpu())
    if sched is not None: sched.step()
    return correct / n, (torch.cat(preds).numpy() if preds else None)

def train_full(variant, n_feat, seed, enc, data, epochs, patience, lr_front, lr_rest, tag, cosine=False):
    model = FullModel(variant, n_feat, seed, enc).to(DEVICE)
    front = list(model.head.front.parameters())
    front_ids = {id(p) for p in front}
    rest = [p for p in model.parameters() if id(p) not in front_ids]
    groups = [{'params': rest, 'lr': lr_rest}] + ([{'params': front, 'lr': lr_front}] if front else [])
    opt = torch.optim.Adam(groups)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if cosine else None
    bs = CFG['strong_batch'] if enc == 'strong' else CFG['enc_batch']
    kw = dict(batch_size=bs, num_workers=CFG['num_workers'], pin_memory=DEVICE == 'cuda')
    tr = DataLoader(data['train'], shuffle=True, generator=torch.Generator().manual_seed(seed), **kw)
    va, te = DataLoader(data['val'], shuffle=False, **kw), DataLoader(data['test'], shuffle=False, **kw)
    eg = lambda: torch.Generator(device=DEVICE).manual_seed(999)
    best, best_state, bad, ep, t0 = -1.0, None, 0, 0, time.time()
    for ep in range(1, epochs + 1):
        te0 = time.time()
        tra, _ = full_epoch(model, tr, opt, None, sched)
        vacc, _ = full_epoch(model, va, None, eg())
        print(f'[{tag}] ep {ep:2d} train {tra:.4f} val {vacc:.4f} ({time.time()-te0:.0f}s)')
        if vacc > best: best, best_state, bad, best_tra = vacc, copy.deepcopy(model.state_dict()), 0, tra
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    tacc, preds = full_epoch(model, te, None, eg())
    return model, dict(variant=variant, seed=seed, lr=lr_front, val_acc=best, test_acc=tacc, train_acc=best_tra, epochs=ep,
                       seconds=round(time.time() - t0, 1), front_params=sum(p.numel() for p in front),
                       total_params=n_trainable(model), dataset='cifar10'), preds

@torch.no_grad()
def extract_angles(model, ds, seed):
    model.eval()
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=CFG['num_workers'])
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    A, Y = [], []
    for x, y in loader:
        A.append(model.angles(x.to(DEVICE), g).cpu()); Y.append(y)
    return torch.cat(A), torch.cat(Y)

def frozen_features(n_feat, enc, data, prev_path, tag):
    cache = f'{OUT}/features/frozen_{tag}.pt'
    for p in [prev_path, cache]:
        if p and os.path.exists(p):
            print(f'[{tag}] loaded frozen features from', p); return torch.load(p)
    ckpt = f'{OUT}/checkpoints/encoder_{tag}.pt'
    if os.path.exists(ckpt):
        enc_model = FullModel('bypass', n_feat, 0, enc).to(DEVICE); enc_model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    else:
        if enc == 'base':
            enc_model, r, _ = train_full('bypass', n_feat, 0, 'base', data, CFG['enc_epochs'], CFG['enc_patience'],
                                         CFG['enc_lr'], CFG['enc_lr'], f'encoder-{tag}')
        else:
            enc_model, r, _ = train_full('bypass', n_feat, 0, 'strong', data, CFG['strong_epochs'], CFG['strong_patience'],
                                         CFG['strong_lr'], CFG['strong_lr'], f'encoder-{tag}', cosine=True)
        torch.save(enc_model.state_dict(), ckpt)
        json.dump(r, open(f'{OUT}/checkpoints/encoder_{tag}_result.json', 'w'), indent=2)
        print(f'[{tag}] phase-1 result:', r)
    train_src = data.get('train_eval', data['train'])          # no augmentation when extracting features
    feats = {k: extract_angles(enc_model, ds, s) for k, ds, s in [('train', train_src, 11), ('val', data['val'], 12), ('test', data['test'], 13)]}
    if (feats['train'][0].std(0) < 1e-3).any():
        print(f'[{tag}] WARNING: some encoder outputs are (near-)constant.')
    torch.save(feats, cache)
    return feats

## Stage 2 — heads on frozen features (CIFAR-10, base encoder): learning rate tuned per variant, 10 seeds
- `headsA`: 4 features. Loads the same frozen features as v1/v2 when `prev_features_4` exists.
- `headsB`: 16 features. Loads the v2 Part-B features when `prev_features_16` exists.

In [ ]:
if CFG['run_heads']:
    featsA = frozen_features(4, 'base', cifar_base, CFG['prev_features_4'], 'base4')
    run_experiment('headsA', VARIANTS_A, head_runner(4, featsA), CFG['eval_seeds'])
    featsB = frozen_features(16, 'base', cifar_base, CFG['prev_features_16'], 'base16')
    run_experiment('headsB', VARIANTS_B, head_runner(16, featsB), CFG['eval_seeds'])
    print('chosen learning rates:', json.dumps({k: lr_choices().get(k) for k in ['headsA', 'headsB']}, indent=1))

## Stage 3 — stronger encoder (item 7): 4 conv layers + BatchNorm, direct input coding, augmentation, 16 features
Tests whether the null result holds when the encoder is far from the ~62% ceiling. Phase 1 trains this encoder once (cosine schedule); its frozen features are then shared by every head.

In [ ]:
if CFG['run_strong']:
    featsS = frozen_features(16, 'strong', cifar_strong, None, 'strong16')
    run_experiment('headsB_strong', VARIANTS_B, head_runner(16, featsS), CFG['eval_seeds'])

## Stage 4 — end-to-end training (item 1)
Encoder and head are trained jointly from scratch. The front-end (circuit or classical core) uses the learning rate tuned for it in Stage 2/3; the encoder and classifier use `enc_lr`. 3 seeds per variant. Each run takes about 35–45 min on GPU and is saved as soon as it finishes.

In [ ]:
def e2e_runner(n_feat, enc, data, cosine):
    def run_one(v, s, lr, stage):
        epochs, pat, lr_rest = ((CFG['e2e_epochs'], CFG['e2e_patience'], CFG['enc_lr']) if enc == 'base'
                                else (CFG['strong_epochs'], CFG['strong_patience'], CFG['strong_lr']))
        _, row, preds = train_full(v, n_feat, s, enc, data, epochs, pat, lr, lr_rest, f'{enc}{n_feat}-{v}-s{s}', cosine)
        return row, preds
    return run_one

def tuned_lr(head_exp):
    return lambda v: lr_choices().get(head_exp, {}).get(v, CFG['default_lr'])

if CFG['run_e2e']:
    run_experiment('e2eA', CFG['e2e_variants_A'], e2e_runner(4, 'base', cifar_base, False), CFG['e2e_seeds'], lr_from=tuned_lr('headsA'))
    run_experiment('e2eB', CFG['e2e_variants_B'], e2e_runner(16, 'base', cifar_base, False), CFG['e2e_seeds'], lr_from=tuned_lr('headsB'))
    if CFG['e2e_strong']:
        run_experiment('e2eB_strong', CFG['e2e_variants_B_strong'], e2e_runner(16, 'strong', cifar_strong, True),
                       CFG['e2e_seeds'], lr_from=tuned_lr('headsB_strong'))

## Stage 5 — Part C: Xu et al. PPF-SQNN on MNIST, completed (item 5)
- **Matched control:** the circuit in the second branch is replaced by a classical core of identical parameter count (Linear n→h + tanh → fixed random h→n → tanh, h·(n+1) = 3·L·n).
- **Per-branch accuracy:** SNN branch alone and second branch alone are logged for every run.
- **Reduction width** 5 / 20 / 60: Linear(784→width) followed by width/5 parallel 5-qubit blocks (or classical cores), testing whether a stronger second branch recovers their ~97%.
- **Logit fusion** `log_softmax(ξ·z_q + (1−ξ)·z_c)` as an alternative to their probability mixture.
- **ξ sweep** with ξ chosen on validation.
- Learning rate tuned per variant on validation (width 5, probability fusion, ξ = 0.8), then reused for all widths.

Unspecified details of their model, fixed here: 128 hidden LIF neurons, 10 time steps, L = 2 circuit layers, learned Linear(784→width) reduction.

In [ ]:
class SFNN(nn.Module):
    def __init__(self, hidden=128, beta=0.9):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid(), init_hidden=True)
        self.fc2 = nn.Linear(hidden, 10)
    def forward(self, spikes):
        snn_utils.reset(self)
        return self.fc2(torch.stack([self.lif1(self.fc1(spikes[t])) for t in range(spikes.shape[0])]).mean(0))

class SecondBranch(nn.Module):
    def __init__(self, core, width=5, n=5, L=2, seed=0):
        super().__init__()
        assert width % n == 0
        self.core_type, self.n, self.G = core, n, width // n
        self.reduce, self.readout = nn.Linear(784, width), nn.Linear(width, 10)
        g = torch.Generator().manual_seed(seed)
        if core == 'quantum':
            self.w = nn.Parameter((torch.rand(self.G, L, n, 3, generator=g) * 2 - 1) * math.pi)
        elif core == 'classical':
            h = max(1, round(3 * L * n / (n + 1)))
            self.lins = nn.ModuleList([nn.Linear(n, h) for _ in range(self.G)])
            self.register_buffer('proj', torch.randn(self.G, h, n, generator=g) / math.sqrt(h))
    def core_params(self):
        if self.core_type == 'quantum': return self.w.numel()
        if self.core_type == 'classical': return sum(p.numel() for p in self.lins.parameters())
        return 0
    def forward(self, x):
        a = math.pi * torch.tanh(self.reduce(x)); n = self.n
        if self.core_type == 'quantum':
            z = torch.cat([xu_circuit(a[:, g * n:(g + 1) * n], self.w[g]) for g in range(self.G)], 1)
        elif self.core_type == 'classical':
            z = torch.cat([torch.tanh(torch.tanh(self.lins[g](a[:, g * n:(g + 1) * n])) @ self.proj[g]) for g in range(self.G)], 1)
        else:
            z = a / math.pi
        return self.readout(z)

class PPF(nn.Module):
    CORE = {'sfnn_only': None, 'ppf_linear': 'identity', 'ppf_classical': 'classical', 'ppf_quantum': 'quantum'}
    def __init__(self, variant, xi, width, fusion, seed):
        super().__init__()
        set_seed(seed)
        self.sfnn = SFNN(CFG_C['hidden'], CFG['beta'])
        core = self.CORE[variant]
        self.second = None if core is None else SecondBranch(core, width, CFG_C['n_block'], CFG_C['q_layers'], seed)
        self.xi, self.fusion = (0.0 if core is None else xi), fusion
    def forward(self, img, gen=None, branches=False):
        x = img.flatten(1)
        zc = self.sfnn(rate_encode(x, CFG_C['num_steps'], gen))
        zq = None if self.second is None else self.second(x)
        if zq is None or self.xi == 0:
            out = F.log_softmax(zc, -1)
        elif self.fusion == 'prob':
            out = torch.log(self.xi * F.softmax(zq, -1) + (1 - self.xi) * F.softmax(zc, -1) + 1e-9)
        else:
            out = F.log_softmax(self.xi * zq + (1 - self.xi) * zc, -1)
        return (out, zc, zq) if branches else out

def ppf_epoch(model, loader, opt=None, gen=None):
    train = opt is not None
    model.train(train); n = c = c_snn = c_sec = 0; preds = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            out, zc, zq = model(x, gen, branches=True); loss = F.nll_loss(out, y)
        if train: opt.zero_grad(); loss.backward(); opt.step()
        p = out.argmax(1); n += y.numel(); c += (p == y).sum().item(); c_snn += (zc.argmax(1) == y).sum().item()
        if zq is not None: c_sec += (zq.argmax(1) == y).sum().item()
        if not train: preds.append(p.cpu())
    sec = c_sec / n if model.second is not None else None
    return c / n, c_snn / n, sec, (torch.cat(preds).numpy() if preds else None)

def train_ppf(variant, seed, lr, xi, width, fusion):
    model = PPF(variant, xi, width, fusion, seed).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    kw = dict(batch_size=CFG_C['batch'], num_workers=CFG['num_workers'], pin_memory=DEVICE == 'cuda')
    tr = DataLoader(mnist['train'], shuffle=True, generator=torch.Generator().manual_seed(seed), **kw)
    va, te = DataLoader(mnist['val'], shuffle=False, **kw), DataLoader(mnist['test'], shuffle=False, **kw)
    eg = lambda: torch.Generator(device=DEVICE).manual_seed(999)
    best, bad, ep, t0 = -1.0, 0, 0, time.time()
    for ep in range(1, CFG_C['epochs'] + 1):
        tra, _, _, _ = ppf_epoch(model, tr, opt)
        vacc, vs, vq, _ = ppf_epoch(model, va, None, eg())
        if vacc > best: best, best_state, bad, best_tra, bvs, bvq = vacc, copy.deepcopy(model.state_dict()), 0, tra, vs, vq
        else:
            bad += 1
            if bad >= CFG_C['patience']: break
    model.load_state_dict(best_state)
    tacc, ts, tq, preds = ppf_epoch(model, te, None, eg())
    return dict(variant=variant, seed=seed, lr=lr, xi=model.xi, width=width, fusion=fusion, train_acc=best_tra, val_acc=best,
                test_acc=tacc, val_acc_snn=bvs, val_acc_second=bvq, test_acc_snn=ts, test_acc_second=tq, epochs=ep,
                seconds=round(time.time() - t0, 1), core_params=0 if model.second is None else model.second.core_params(),
                total_params=n_trainable(model), dataset='mnist'), preds

def ppf_runner(xi, width, fusion):
    return lambda v, s, lr, stage: train_ppf(v, s, lr, xi, width, fusion)

if CFG['run_partC']:
    VC = ['ppf_linear', 'ppf_classical', 'ppf_quantum']
    base_extra = f"w5|prob|xi{CFG_C['xi']}"
    for v in ['sfnn_only'] + VC:                                     # tune once, reuse everywhere
        choose_lr('partC', v, ppf_runner(CFG_C['xi'], 5, 'prob'), extra=base_extra, tune_exp='partC_tune')
    lr_c = lambda v: lr_choices()['partC'][v]
    run_experiment('partC_sfnn', ['sfnn_only'], ppf_runner(0.0, 5, 'prob'), CFG_C['seeds'], lr_from=lr_c)
    for w in CFG_C['widths']:
        run_experiment(f'partC_w{w}_prob', VC, ppf_runner(CFG_C['xi'], w, 'prob'), CFG_C['seeds'],
                       extra=f"w{w}|prob|xi{CFG_C['xi']}", lr_from=lr_c)
    for w in CFG_C['logit_widths']:
        run_experiment(f'partC_w{w}_logit', VC, ppf_runner(CFG_C['xi'], w, 'logit'), CFG_C['seeds'],
                       extra=f"w{w}|logit|xi{CFG_C['xi']}", lr_from=lr_c)
    if CFG_C['run_xi_sweep']:
        for xi in CFG_C['xi_grid']:
            run_experiment('partC_xisweep', ['ppf_classical', 'ppf_quantum'], ppf_runner(xi, 5, 'prob'),
                           CFG_C['sweep_seeds'], extra=f'w5|prob|xi{xi}', lr_from=lr_c)

In [ ]:
if CFG['run_partC']:
    rows = []
    for f in sorted(os.listdir(OUT)):
        if f.startswith('runs_partC_w') or f == 'runs_partC_sfnn.csv':
            d = pd.read_csv(f'{OUT}/{f}'); rows.append(d[d.stage == 'eval'])
    pc = pd.concat(rows)
    br = (pc.groupby(['experiment', 'variant']).agg(test=('test_acc', 'mean'), snn_branch=('test_acc_snn', 'mean'),
                                                    second_branch=('test_acc_second', 'mean'), train=('train_acc', 'mean'),
                                                    n=('seed', 'size')) * [100, 100, 100, 100, 1]).round(2)
    display(br)
    print('Per-branch accuracy shows whether the fused prediction is limited by the second branch.')

## Stage 6 — statistics, paper tables and figures

In [ ]:
if CFG['run_aggregate']:
    importlib.reload(mpt)
    res = mpt.main(OUT, delta=CFG['delta'], n_boot=CFG['n_boot'])
    if 'comparisons' in res and not res['comparisons'].empty:
        display(res['comparisons'][['experiment', 'target', 'control', 'n_seeds', 'delta', 'boot_lo', 'boot_hi',
                                    'p_holm', 'p_tost', 'verdict']])
    print('\nFiles:', sorted(os.listdir(f'{OUT}/paper')))

## Reporting checklist
- **Methods:** both datasets and splits exactly as stated at the top (split indices are in `splits/`); rate coding with 25 steps (base) / direct coding with 6 steps (strong) / rate coding with 10 steps (MNIST); learning-rate grid and tuning seeds; 10 evaluation seeds for heads and Part C, 3 for end-to-end; exact state-vector simulation (no shot noise), cross-checked against Qiskit.
- **Statistics:** report the `comparisons` table. Claim **EQUIVALENT** only where TOST p < 0.05 at ±0.5 pts; claim BETTER/WORSE only where the Holm-corrected p < 0.05 **and** the bootstrap CI over test images excludes 0.
- **Part C:** list every assumed detail of Xu et al.'s model; report per-branch accuracy and the validation-selected ξ.
- **Reproducibility:** release `env.json`, `requirements.txt`, `runs_*.csv`, `preds/`, `splits/`, `lr_choice.json`, `circuit_diagnostics.json`, and `make_paper_tables.py`. Every number in the paper can then be regenerated with `python make_paper_tables.py --results qsnn_v3_results`.